# 04 · ROI extraction (detection → classification dataset)

Converts a YOLO detection dataset into an `ImageFolder` classification dataset by cropping every annotated defect region (ROI) into `train/ valid/ test/ × {Black_Spot, Cell_Scratch, Cell_Difference, Micro_Crack}`.

| | |
|---|---|
| **Input** | YOLO-format dataset (`YOLO_DATA_DIR`) |
| **Output** | `OUTPUT_DIR/{train,valid,test}/<class>/*.jpg` |
| **Runs on** | CPU |

> **Notes**
> - `convert_yolo_to_classification` saves tight bounding-box crops. `smart_solar_preprocess` previews a square crop with a ×1.5 context margin, bicubic resize to 224×224 and mild CLAHE.
> - Classifier training lives in notebook `05`; the earlier duplicate of it that was at the end of this notebook has been removed.
> - Crops are never committed (see `DATA_NOTICE.md`).

# YOLO → Classification Dataset Converter
## Convert YOLO Detection Format to ImageFolder Classification Format

**What this does:**
1. Reads YOLO format annotations (class_id + bounding boxes)
2. Extracts defect regions OR uses whole image
3. Organizes into folders by class for PyTorch ImageFolder

**Output Structure:**
```
Classification_Dataset/
├── train/
│   ├── Black_Spot/
│   ├── Cell_Scratch/
│   ├── Cell_Difference/
│   └── Micro_Crack/
├── valid/
└── test/
```

---
## Step 1: Setup & Imports

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive mounted!")

In [ ]:
import os
import shutil
import cv2
import numpy as np
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt

print("✅ All imports successful!")

---
## Step 2: Configuration

In [ ]:
# =============================================================================
# CONFIGURATION - 🔴 UPDATE THESE PATHS
# =============================================================================

# Source: Your YOLO format dataset
YOLO_DATA_DIR = Path('/content/drive/MyDrive/inspectra/detection_dataset_raw')

# Destination: Where to create classification dataset
OUTPUT_DIR = Path('/content/drive/MyDrive/inspectra/classification_dataset')

# Class names (in order matching class IDs: 0, 1, 2, 3)
CLASS_NAMES = [
    'Black_Spot',      # Class 0
    'Cell_Scratch',    # Class 1
    'Cell_Difference', # Class 2
    'Micro_Crack'      # Class 3
]

# Mode selection:
# 'whole_image' = Copy entire image to class folder (simpler, for multi-label later)
# 'crop_defects' = Crop individual defect regions from bounding boxes
MODE = 'crop_defects'  # 🔴 CHANGE TO 'crop_defects' IF YOU WANT CROPPED REGIONS

# If multiple defects in one image:
# 'first' = Use only the first defect class
# 'dominant' = Use the class with most bounding boxes
MULTI_DEFECT_STRATEGY = 'dominant'

print("="*80)
print("⚙️ CONFIGURATION")
print("="*80)
print(f"Source (YOLO): {YOLO_DATA_DIR}")
print(f"Output (Classification): {OUTPUT_DIR}")
print(f"Mode: {MODE}")
print(f"Classes: {CLASS_NAMES}")
print(f"Multi-defect strategy: {MULTI_DEFECT_STRATEGY}")
print("="*80)

---
## Step 3: Verify YOLO Data Structure

In [ ]:
# Also search for images in sub-folders
for split in ['train', 'valid', 'test']:
    split_dir = YOLO_DATA_DIR / split
    if split_dir.exists():
        # Search the main folder and the images/ sub-folder
        images = list(split_dir.glob('*.jpg')) + list(split_dir.glob('images/*.jpg')) + \
                 list(split_dir.glob('*.png')) + list(split_dir.glob('images/*.png'))

        txt_files = [f for f in split_dir.glob('*.txt')] + \
                    [f for f in split_dir.glob('labels/*.txt')]

        # Filter out classes.txt
        txt_files = [f for f in txt_files if f.name != 'classes.txt']

        print(f"📁 {split:8s}: {len(images):4d} images, {len(txt_files):4d} annotation files")
    else:
        print(f"❌ {split:8s}: NOT FOUND")

---
## Step 4: Create Output Directory Structure

In [ ]:
# Create output directories
print("🏗️ Creating output directory structure...\n")

for split in ['train', 'valid', 'test']:
    for class_name in CLASS_NAMES:
        class_dir = OUTPUT_DIR / split / class_name
        class_dir.mkdir(parents=True, exist_ok=True)
        print(f"✅ Created: {class_dir}")

print(f"\n✅ Output structure ready at: {OUTPUT_DIR}")

---
## Step 5: Conversion Functions

In [ ]:
def parse_yolo_annotation(txt_file):
    """
    Parse a YOLO annotation file

    Returns:
        List of tuples: (class_id, x_center, y_center, width, height)
    """
    annotations = []

    if not txt_file.exists():
        return annotations

    with open(txt_file, 'r') as f:
        lines = f.readlines()

    for line in lines:
        line = line.strip()
        if not line:
            continue

        parts = line.split()
        if len(parts) >= 5:
            try:
                class_id = int(parts[0])
                x_center = float(parts[1])
                y_center = float(parts[2])
                width = float(parts[3])
                height = float(parts[4])
                annotations.append((class_id, x_center, y_center, width, height))
            except ValueError:
                continue

    return annotations


def yolo_to_pixel_coords(annotation, img_width, img_height):
    """
    Convert YOLO normalized coordinates to pixel coordinates

    Args:
        annotation: (class_id, x_center, y_center, width, height) in normalized coords
        img_width, img_height: Image dimensions in pixels

    Returns:
        (x1, y1, x2, y2) in pixels
    """
    class_id, x_center, y_center, width, height = annotation

    # Convert to pixels
    x_center_px = x_center * img_width
    y_center_px = y_center * img_height
    width_px = width * img_width
    height_px = height * img_height

    # Get bounding box corners
    x1 = int(x_center_px - width_px / 2)
    y1 = int(y_center_px - height_px / 2)
    x2 = int(x_center_px + width_px / 2)
    y2 = int(y_center_px + height_px / 2)

    # Clip to image bounds
    x1 = max(0, x1)
    y1 = max(0, y1)
    x2 = min(img_width, x2)
    y2 = min(img_height, y2)

    return x1, y1, x2, y2


def get_dominant_class(annotations):
    """
    Get the dominant class from multiple annotations

    Returns:
        class_id of dominant class
    """
    if not annotations:
        return None

    # Count occurrences of each class
    class_counts = {}
    for ann in annotations:
        class_id = ann[0]
        class_counts[class_id] = class_counts.get(class_id, 0) + 1

    # Return most frequent class
    return max(class_counts.items(), key=lambda x: x[1])[0]


print("✅ Conversion functions defined!")

---
## Step 6: Convert Dataset

In [ ]:
def convert_yolo_to_classification(yolo_split_dir, output_dir, split_name, mode='crop_defects'):
    """
    Variant that looks for annotations in separate labels/ folders
    """
    print(f"\n{'='*80}")
    print(f"🔄 Converting {split_name.upper()} set ({mode} mode)")
    print(f"{'='*80}\n")

    # 1. Locate the images folder and the labels folder
    img_dir = yolo_split_dir / 'images' if (yolo_split_dir / 'images').exists() else yolo_split_dir
    label_dir = yolo_split_dir / 'labels' if (yolo_split_dir / 'labels').exists() else yolo_split_dir

    # 2. Collect all image files
    image_files = []
    for ext in ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.PNG']:
        image_files.extend(list(img_dir.glob(ext)))

    if not image_files:
        print(f"❌ No images found in {img_dir}")
        return None

    stats = {
        'total_images': len(image_files),
        'images_with_annotations': 0,
        'images_without_annotations': 0,
        'crops_created': 0,
        'class_counts': {name: 0 for name in CLASS_NAMES}
    }

    # 3. Process each image
    for img_file in tqdm(image_files, desc=f"Processing {split_name}"):
        img = cv2.imread(str(img_file))
        if img is None: continue

        img_h, img_w = img.shape[:2]

        # Key change: look up the annotation file in the labels folder
        txt_file = label_dir / f"{img_file.stem}.txt"
        annotations = parse_yolo_annotation(txt_file)

        if not annotations:
            stats['images_without_annotations'] += 1
            continue

        stats['images_with_annotations'] += 1

        # Extract the defect regions (ROIs)
        if mode == 'crop_defects':
            for idx, ann in enumerate(annotations):
                class_id = ann[0]
                if class_id < 0 or class_id >= len(CLASS_NAMES): continue

                class_name = CLASS_NAMES[class_id]
                stats['class_counts'][class_name] += 1

                # Compute the pixel coordinates
                x1, y1, x2, y2 = yolo_to_pixel_coords(ann, img_w, img_h)

                # Crop the defect region
                cropped = img[y1:y2, x1:x2]

                if cropped.size == 0: continue

                # Save the cropped image
                output_filename = f"{img_file.stem}_roi_{idx}.jpg"
                save_path = output_dir / split_name / class_name / output_filename
                cv2.imwrite(str(save_path), cropped)
                stats['crops_created'] += 1

    # Print the results for this split
    print(f"\n✅ {split_name.upper()} Complete!")
    print(f"   Images with annotations: {stats['images_with_annotations']}")
    print(f"   Crops created: {stats['crops_created']}")
    return stats

# Run the conversion
all_stats = {}
for split in ['train', 'valid', 'test']:
    yolo_split_dir = YOLO_DATA_DIR / split
    if yolo_split_dir.exists():
        all_stats[split] = convert_yolo_to_classification(yolo_split_dir, OUTPUT_DIR, split, mode=MODE)

---
## Step 7: Summary & Verification

In [ ]:
print("\n" + "="*80)
print("📊 CONVERSION SUMMARY")
print("="*80 + "\n")

# Overall statistics
total_samples = 0
class_totals = {name: 0 for name in CLASS_NAMES}

for split, stats in all_stats.items():
    if MODE == 'whole_image':
        split_total = stats['images_with_annotations']
    else:
        split_total = stats['crops_created']

    total_samples += split_total

    for class_name, count in stats['class_counts'].items():
        class_totals[class_name] += count

print(f"Total samples created: {total_samples}\n")
print(f"Overall class distribution:")
for class_name, count in class_totals.items():
    percentage = (count / total_samples * 100) if total_samples > 0 else 0
    print(f"   {class_name:20s}: {count:5d} samples ({percentage:5.1f}%)")

print(f"\n✅ Classification dataset ready at: {OUTPUT_DIR}")
print(f"\n💡 You can now use this with PyTorch ImageFolder!")
print("="*80)

---
## Step 8: Visualize Sample from Each Class

In [ ]:
# Visualize one sample from each class
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle('Sample from Each Defect Class', fontsize=16, fontweight='bold')

for idx, class_name in enumerate(CLASS_NAMES):
    class_dir = OUTPUT_DIR / 'train' / class_name

    # Get first image in class
    images = list(class_dir.glob('*.jpg')) + list(class_dir.glob('*.png'))

    if images:
        img = cv2.imread(str(images[0]), cv2.IMREAD_GRAYSCALE)

        ax = axes[idx]
        ax.imshow(img, cmap='gray')
        ax.set_title(class_name, fontsize=12, fontweight='bold')
        ax.axis('off')
    else:
        axes[idx].text(0.5, 0.5, 'No samples', ha='center', va='center')
        axes[idx].set_title(class_name, fontsize=12)
        axes[idx].axis('off')

plt.tight_layout()
plt.show()